In [1]:
from ibl_info.prepare_data_pid import (
    get_congruent_incongruent_intervals,
    get_window,
    get_contrast_intervals,
)
from ibl_info.selective_decomposition import filter_eids
import numpy as np
from brainbox.population.decode import get_spike_counts_in_bins
from one.api import ONE
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.singlecell import bin_spikes2D, firing_rate
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
import os

/Users/kschille/micromamba/envs/info-decom/lib/python3.12/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [2]:
from ibl_info.utils import check_config

In [2]:
important_regions = np.asarray(
    [
        "VISp",
        "MOs",
        "SSp-ul",
        "ACAd",
        "PL",
        "CP",
        "VPM",
        "MG",
        "LGd",
        "ZI",
        "SNr",
        "MRN",
        "SCm",
        "PAG",
        "APN",
        "RN",
        "PPN",
        "PRNc",
        "PRNr",
        "GRN",
        "IRN",
        "PGRN",
        "CUL4 5",
        "SIM",
        "IP",
    ]
)

one = ONE()
unit_df = bwm_units(one)
for region in important_regions:
    selective_eids = filter_eids(unit_df, region)
    break

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [3]:
session_id = selective_eids[0]

In [4]:
pids, probes = one.eid2pid(session_id)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

trials, mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)
trials = trials[mask]

# for now we are looking at just (stimulus interval)
# we know the order
intervals_by_congruency, _ = get_congruent_incongruent_intervals(trials, "stim")

In [5]:
def subdivide_interval(interval_start: float, interval_end: float, width: float):
    """
    Subdivides a time interval [interval_start, interval_end] into bins of 'width' milliseconds.

    Args:
        interval_start: The start time of the main interval in seconds.
        interval_end: The end time of the main interval in seconds.
        width: The size of each sub-interval bin in milliseconds.

    Returns:
        A list of tuples, where each tuple represents a sub-interval (start_time, end_time) in seconds.
    """
    if interval_start >= interval_end:
        raise ValueError("interval_start must be less than interval_end.")
    if width <= 0:
        raise ValueError("width must be a positive number.")

    bin_seconds = width / 1000.0  # Convert bin size from ms to seconds
    intervals = []

    current_time = interval_start
    epsilon = 1e-6  # 1 nanosecond, a common small value for time comparisons
    while current_time < interval_end - epsilon:
        end_of_bin = min(current_time + bin_seconds, interval_end)
        intervals.append((current_time, end_of_bin))
        current_time = end_of_bin

    return intervals

In [6]:
# do check if cluster acronyms are in the regions provided
brainreg = BrainRegions()
beryl_regions = brainreg.acronym2acronym(clusters["acronym"], mapping="Beryl")

# find all clusters in region (where region can be a list of regions)
region_mask = np.isin(beryl_regions, region)
actual_regions = region
n_units = np.sum(region_mask)

# find all spikes in those clusters
spike_mask = np.isin(spikes["clusters"], clusters[region_mask].index)
times_masked = spikes["times"][spike_mask]
clusters_masked = spikes["clusters"][spike_mask]
# record cluster uuids
idxs_used = np.unique(clusters_masked)
clusters_uuids = list(clusters.iloc[idxs_used]["uuids"])

In [7]:
intervals_by_congruency[0].shape

(263, 2)

In [32]:
tiled_intervals = []
for idx in range(intervals_by_congruency[0].shape[0]):
    tiled_intervals.append(
        subdivide_interval(
            interval_start=intervals_by_congruency[0][idx][0],
            interval_end=intervals_by_congruency[0][idx][1],
            width=20,
        )
    )

In [33]:
tiled_intervals = np.asarray(tiled_intervals)

In [34]:
tiled_intervals.shape

(263, 5, 2)

In [35]:
spike_data = []

for tiles in range(tiled_intervals.shape[1]):
    binned, _ = get_spike_counts_in_bins(
        spike_times=times_masked,
        spike_clusters=clusters_masked,
        intervals=tiled_intervals[:, tiles, :],
    )
    spike_data.append(binned)
spike_data = np.asarray(spike_data)

In [42]:
X = np.random.choice([0, 1], size=(100))

In [44]:
Y = np.zeros_like(X)

In [46]:
from ibl_info.measures.information_measures import corrected_mutual_information

In [48]:
corrected_mutual_information(X, Y)

np.float64(0.0)